In [1]:
import re
import json
import argparse
from pathlib import Path

# Movesets

In [8]:
INPUT_PATH = Path(
    "/home/yan/Documents/GitHub/competitive-poke-ai123/databases/smogon_metagame_data/movesets_gen9championsvgc2026regmb-1760.txt"
)

formato = "regmb"
OUTPUT_JSONL = Path(f"pokemon_moveset_{formato}.jsonl")

SECTIONS = {"Abilities", "Items", "Spreads", "Moves", "Teammates"}

METRICS = {
    "Raw count": "raw_count",
    "Avg. weight": "avg_weight",
    "Viability Ceiling": "viability_ceiling",
}

PERCENT_RE = re.compile(r"^(.*?)\s+(\d+(?:[.,]\d+)?)\s*%$")


def read_txt(path):
    """Lê o arquivo TXT com fallback de encoding."""
    for encoding in ("utf-8", "latin-1", "cp1252"):
        try:
            return Path(path).read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError("Não foi possível decodificar o arquivo.")


def parse_value(line):
    """Extrai nome e porcentagem de uma linha."""
    match = PERCENT_RE.match(line.strip())

    if match:
        name = match.group(1).strip()
        percentage = float(match.group(2).replace(",", "."))

        return {"name": name, "percentage": percentage}

    return {"name": line.strip(), "percentage": None}


def parse_txt(text):
    """Converte o TXT em registros estruturados."""
    pokemon_data = []
    current = None
    current_section = None

    for line in text.splitlines():
        # Remove bordas da tabela ASCII
        line = line.strip()

        if not line.startswith("|"):
            continue

        line = line.strip("|").strip()

        if not line:
            continue

        # Identifica início de uma nova seção
        if line in SECTIONS:
            current_section = line.lower()

            if current is not None:
                current["sections"].setdefault(current_section, [])

            continue

        # Ignora linhas de separação ou títulos vazios
        if line.startswith("-"):
            continue

        # Identifica métricas gerais
        metric_found = False

        for metric, key in METRICS.items():
            prefix = metric + ":"

            if line.startswith(prefix):
                if current is not None:
                    value = line[len(prefix) :].strip()

                    try:
                        value = float(value.replace(",", "."))
                        if value.is_integer():
                            value = int(value)
                    except ValueError:
                        pass

                    current["stats"][key] = value

                metric_found = True
                break

        if metric_found:
            continue

        # Identifica um novo Pokémon (Nomes aparecem isolados, sem porcentagem ou métrica)
        if PERCENT_RE.match(line) is None:
            # Pode ser uma métrica que não foi reconhecida
            if ":" in line:
                continue

            current = {"pokemon": line, "stats": {}, "sections": {}}

            pokemon_data.append(current)
            current_section = None
            continue

        # Extrai os dados da seção atual
        if current is not None and current_section is not None:
            item = parse_value(line)
            current["sections"][current_section].append(item)

    return pokemon_data


def generate_rag_documents(records):
    """Gera documentos independentes por Pokémon e categoria."""
    documents = []

    for pokemon in records:
        name = pokemon["pokemon"]

        # Documento de estatísticas gerais
        stats = pokemon["stats"]

        if stats:
            content = f"Pokémon: {name}\nEstatísticas gerais:\n"

            for key, value in stats.items():
                content += f"{key}: {value}\n"

            documents.append({
                "id": f"{name}_stats",
                "pokemon": name,
                "category": "stats",
                "content": content.strip(),
                "metadata": stats,
            })

        # Documento independente para cada categoria
        for category, entries in pokemon["sections"].items():
            if not entries:
                continue

            content = (
                f"Pokémon: {name}\n"
                f"Categoria: {category}\n"
                f"Dados de {category}:\n"
            )

            for entry in entries:
                if entry["percentage"] is not None:
                    content += f"- {entry['name']}: {entry['percentage']}%\n"
                else:
                    content += f"- {entry['name']}\n"

            documents.append({
                "id": f"{name}_{category}",
                "pokemon": name,
                "category": category,
                "content": content.strip(),
                "metadata": {"entries": entries},
            })

    return documents


def save_jsonl(documents, output_path):
    """Salva um documento por linha."""
    with open(output_path, "w", encoding="utf-8") as file:
        for document in documents:
            file.write(json.dumps(document, ensure_ascii=False) + "\n")


def main():
    # Garante que as pastas de destino existam, caso passe um diretório completo
    OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

    text = read_txt(INPUT_PATH)
    records = parse_txt(text)
    documents = generate_rag_documents(records)


    # Salva documentos para RAG
    save_jsonl(documents, OUTPUT_JSONL)

    print(f"Pokémon encontrados: {len(records)}")
    print(f"Documentos gerados: {len(documents)}")
    print(f"JSONL salvo em: {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Pokémon encontrados: 277
Documentos gerados: 1662
JSONL salvo em: pokemon_moveset_regmb.jsonl


# Leads

In [ ]:
formato = "regmb"

INPUT = Path("/home/yan/Documents/GitHub/competitive-poke-ai123/databases/smogon_metagame_data/leads_gen9championsvgc2026regmb-1760.txt")
OUTPUT = Path(f"pokemon_lead_{formato}.jsonl")

text = INPUT.read_text(encoding="utf-8", errors="replace")

# Extrair total de leads
match = re.search(r"Total leads:\s*(\d+)", text)
total_leads = int(match.group(1)) if match else None

# Padrão das linhas da tabela
pattern = re.compile(
    r"\|\s*(\d+)\s*\|\s*(.*?)\s*\|\s*"
    r"([\d.]+)%\s*\|\s*(\d+)\s*\|\s*([\d.]+)%\s*\|"
)

count = 0

with OUTPUT.open("w", encoding="utf-8") as file:

    for line in text.splitlines():

        match = pattern.search(line)

        if not match:
            continue

        rank, pokemon, usage, raw, percentage = match.groups()

        document = {
            "rank": int(rank),
            "pokemon": pokemon.strip(),
            "usage_percent": float(usage),
            "raw": int(raw),
            "percentage": float(percentage),
            "total_leads": total_leads
        }

        file.write(
            json.dumps(document, ensure_ascii=False) + "\n"
        )

        count += 1

print(f"Conversão concluída!")
print(f"Pokémon extraídos: {count}")
print(f"Arquivo gerado: {OUTPUT}")


Conversão concluída!
Pokémon extraídos: 308
Arquivo gerado: pokemon_lead.jsonl


# Geral

In [ ]:
formato = "regmb"

INPUT = Path("/home/yan/Documents/GitHub/competitive-poke-ai123/databases/smogon_metagame_data/uso_geral_gen9championsvgc2026regmb-1760.txt")
OUTPUT = Path(f"pokemon_geral_{formato}.jsonl")


def extract_dataset_metadata(text):

    battles_match = re.search(
        r"Total battles:\s*([\d,]+)",
        text
    )

    weight_match = re.search(
        r"Avg\.\s*weight/team:\s*([\d.]+)",
        text
    )

    return {
        "total_battles": (
            int(battles_match.group(1).replace(",", ""))
            if battles_match else None
        ),
        "avg_weight_team": (
            float(weight_match.group(1))
            if weight_match else None
        )
    }


def extract_pokemon(text):

    pattern = re.compile(
        r"\|\s*(\d+)\s*\|"          # Rank
        r"\s*(.*?)\s*\|"            # Pokemon
        r"\s*([\d.]+)%\s*\|"        # Usage %
        r"\s*(\d+)\s*\|"            # Raw
        r"\s*([\d.]+)%\s*\|"        # Raw %
        r"\s*(\d+)\s*\|"            # Real
        r"\s*([\d.]+)%\s*\|"        # Real %
    )

    records = []

    for line in text.splitlines():

        match = pattern.search(line)

        if not match:
            continue

        rank, pokemon, usage, raw, raw_pct, real, real_pct = (
            match.groups()
        )

        records.append({
            "rank": int(rank),
            "pokemon": pokemon.strip(),
            "usage_percent": float(usage),
            "raw": int(raw),
            "raw_percent": float(raw_pct),
            "real": int(real),
            "real_percent": float(real_pct)
        })

    return records


def convert_to_jsonl(input_path, output_path):

    text = input_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    metadata = extract_dataset_metadata(text)
    pokemon_records = extract_pokemon(text)

    with output_path.open("w", encoding="utf-8") as file:

        for pokemon in pokemon_records:

            document = {
                **pokemon,
                **metadata
            }

            file.write(
                json.dumps(
                    document,
                    ensure_ascii=False
                ) + "\n"
            )

    print("Conversão concluída!")
    print(f"Pokémon extraídos: {len(pokemon_records)}")
    print(f"Total de batalhas: {metadata['total_battles']}")
    print(f"Arquivo gerado: {output_path}")


if __name__ == "__main__":

    convert_to_jsonl(INPUT, OUTPUT)
